In [1]:
import os
os.environ["OPENAI_API_KEY"] = "sk-...."

In [2]:
!pip install -q requests beautifulsoup4 html-to-markdown pypdf openai chromadb ## in requirements.txt la un mediu de jupterlab similar cu cel de la Cap. 11-12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69

## Config

In [21]:
import time
import requests, pandas as pd
from bs4 import BeautifulSoup

KEYWORDS = "Machine Learning Engineer"
LOCATION = "Romania"
MAX_JOBS = 100
CV_PATH = "CV.pdf"
DB_PATH = "chroma_db"

SEARCH = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
JOB = "https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{}"
FILTERS = {"f_TPR": "r604800", "f_E": "4", "f_JT": "F"} # past week, mid-senior, full-time
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                          "aPPLEwEBkIT/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36"}


def get(url, **params):
  time.sleep(2)
  r = requests.get(url, params=params, headers=HEADERS, timeout=30)
  return r.text if r.ok else ""

def txt(node, sel):
  el = node.select_one(sel)
  return el.get_text(strip=True) if el else ""



## Scrape listings

In [7]:
cards = []
for start in range(0, MAX_JOBS + 10, 10):
  html = get(SEARCH, keywords=KEYWORDS, location=LOCATION, start = start, **FILTERS)
  batch = BeautifulSoup(html, "html.parser").select("div.base-card")
  cards += batch


In [11]:
cards[:3]

[<div class="base-card relative w-full hover:no-underline focus:no-underline base-card--link base-search-card base-search-card--link job-search-card" data-column="1" data-entity-urn="urn:li:jobPosting:4460384940" data-impression-id="jobs-search-result-0" data-reference-id="QPPmfZVeJ/TaGec6bapYeQ==" data-row="1" data-tracking-id="Xg2kuZ6mbqOd8d41WuW+ew==">
 <a class="base-card__full-link absolute top-0 right-0 bottom-0 left-0 p-0 z-[2] outline-offset-[4px]" data-tracking-client-ingraph="" data-tracking-control-name="public_jobs_jserp-result_search-card" data-tracking-will-navigate="" href="https://ro.linkedin.com/jobs/view/machine-learning-engineer-at-hipo-ro-4460384940?position=1&amp;pageNum=0&amp;refId=QPPmfZVeJ%2FTaGec6bapYeQ%3D%3D&amp;trackingId=Xg2kuZ6mbqOd8d41WuW%2Bew%3D%3D">
 <span class="sr-only">
               
         
         Machine Learning Engineer
       
       
           </span>
 </a>
 <div class="search-entity-media">
 <img alt="" class="artdeco-entity-image artdec

In [17]:
rows = []
for c in cards:
  link = c.select_one("a.base-card__full-link")
  url = link["href"].split("?")[0]
  rows.append({
      "id": url.rsplit("-", 1)[-1],
      "url": url,
      "title": txt(c, "h3"),
      "company": txt(c, "h4"),
      "location": txt(c, "span.job-search-card__location")

  })

  jobs = (pd.DataFrame(rows)
        .drop_duplicates(["title", "company"])
        .head(MAX_JOBS)
        .reset_index(drop=True)
  )

In [18]:
jobs

,id,url,title,company,location
0,4460384940,https://ro.linkedin.com/jobs/view/machine-lear...,Machine Learning Engineer,Hipo.ro,"Ilfov, Romania"
1,4450710736,https://ro.linkedin.com/jobs/view/machine-lear...,Machine Learning Engineer,AllCloud,"Bucharest, Romania"
2,4458374567,https://ro.linkedin.com/jobs/view/machine-lear...,Machine Learning Engineer,Siemens Energy,"Bucharest, Bucharest, Romania"
3,4459678536,https://ro.linkedin.com/jobs/view/ai-data-scie...,AI Data Science Engineer,Luxoft,Bucharest Metropolitan Area
4,4457789326,https://ro.linkedin.com/jobs/view/artificial-i...,Artificial Intelligence Engineer,Extia,"Bucharest, Romania"
...,...,...,...,...,...
90,4462130921,https://ro.linkedin.com/jobs/view/software-eng...,Software Engineer (Backend) Python + Django,PDQ,"Bucharest, Bucharest, Romania"
91,4441956624,https://ro.linkedin.com/jobs/view/software-eng...,Software Engineer,Riverbed Technology,"Cluj-Napoca, Cluj, Romania"
92,4462400881,https://ro.linkedin.com/jobs/view/python-devel...,Python Developer,Luxoft,"Bucharest, Romania"
93,4403639230,https://ro.linkedin.com/jobs/view/software-bui...,Software Builder / AI-Native Team,BMW TechWorks Romania,"Cluj-Napoca, Cluj, Romania"


In [20]:
jobs.iloc[91].url

'https://ro.linkedin.com/jobs/view/software-engineer-at-riverbed-technology-4441956624'

In [28]:
from html_to_markdown import convert

def description(job_id):
  html = get(JOB.format(job_id))
  div = BeautifulSoup(html, "html.parser").select_one("div.show-more-less-html__markup")
  if not div:
    return ""
  out = convert(div.decode_contents())
  return getattr(out, 'content', out)

descs = []
for i, jid in enumerate(jobs.id, 1):
  descs.append(description(jid))
  print(f"{i}/{len(jobs)}", end="\r")

jobs["description"] = descs
jobs = jobs[jobs.description != ""].reset_index(drop=True)
print(f"\n{len(jobs)} jobs with a description.")

95/95
95 jobs with a description.


## Read CV

In [29]:
from pypdf import PdfReader

cv = "/n".join(p.extract_text() or "" for p in PdfReader(CV_PATH).pages)
print(cv[:500])

Ionel-Gabriel Zahia
Personal Data
City: Bucharest, Romania
Mobile: +40 762 302 642
Email: ionel.zahia@gmail.com
About
I’m an AI Principal Engineer with a strong foundation in Data Science, Software Engine-
ering, and AI systems design. My expertise spans the full machine learning lifecycle, from
data engineering and model development to deployment and monitoring, with a focus on
designing, training, and deploying machine learning and deep learning models across struc-
tured data, time series, im


## Embeddings for jobs

In [30]:
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def embed(text):
  r = openai_client.embeddings.create(model="text-embedding-3-large", input=text)
  return r.data[0].embedding

In [33]:
import chromadb
db = chromadb.PersistentClient(path=DB_PATH)
col = db.get_or_create_collection("jobs", metadata={"hnsw:space": "cosine"})

stored = set(col.get(ids=jobs.id.tolist())["ids"])
new = jobs[~jobs.id.isin(stored)]
print(f"{len(new)} new, {len(jobs) - len(new)} already stored")

if len(new):
  col.upsert(
      ids = new.id.tolist(),
      embeddings=[embed(d) for d in new.description],
      documents = new.description.tolist(),
      metadatas = new[["title", "company", "location", "url"]].to_dict("records")
  )

print(f"{col.count()} jobs in the database.")

95 new, 95 - 95 already stored
95 jobs in the database.


## Rank jobs against CV

In [35]:
hits = col.query(query_embeddings=[embed(cv)], n_results=10)

ranked = (pd.DataFrame(hits["metadatas"][0]).assign(score=[1 - d for d in hits["distances"][0]]))

for _, r in ranked.iterrows():
  print(f"{r.score:.0%} {r.title} @ {r.company} ({r.location})")
  print(f"  {r.url}")


65% Senior Python AI Developer @ Luxoft (Bucharest Metropolitan Area)
  https://ro.linkedin.com/jobs/view/senior-python-ai-developer-at-luxoft-4459812317
62% Python Developer @ Luxoft (Bucharest, Romania)
  https://ro.linkedin.com/jobs/view/python-developer-at-luxoft-4462400881
61% Senior Python Developer @ Luxoft Romania (Bucharest, Romania)
  https://ro.linkedin.com/jobs/view/senior-python-developer-at-luxoft-romania-4462185204
60% Generative AI Engineer @ Cognizant (Romania)
  https://ro.linkedin.com/jobs/view/generative-ai-engineer-at-cognizant-4448027975
59% Digital & AI Transformation Consultant (m/f/x) @ Mitsui Chemicals Group (Cluj-Napoca, Cluj, Romania)
  https://ro.linkedin.com/jobs/view/digital-ai-transformation-consultant-m-f-x-at-mitsui-chemicals-group-4399694891
58% AI Engineer @ NTT DATA Europe & Latam (Sibiu, Sibiu, Romania)
  https://ro.linkedin.com/jobs/view/ai-engineer-at-ntt-data-europe-latam-4448001188
58% Senior Generative AI Engineer @ BMW TechWorks Romania (Cluj